# 07 — Multi-agent: supervisor, tools-that-are-agents, and handoffs

**What you'll learn**

- Why more agents is a cost, not a default — the two shapes worth knowing (a supervisor that calls agents-as-tools, and a conversational handoff), and when isolation or a different toolset justifies either
- Agent-as-tool: wrap a least-privilege specialist (a role-scoped toolset + `run_agent`) as one callable `Tool` a supervisor invokes, and watch the nested call in the trace
- Private vs shared worker context: the token cost of handing a worker one focused question versus forwarding the whole conversation
- Handoff: transfer the running conversation to a differently-instructed agent that takes over the loop, by passing the message list forward — and who owns the loop afterward
- The honest measurement: single-agent vs supervisor+worker on the same ticket, where the extra agent usually buys a lossy boundary, not a better answer

*Time: ~3 min on a first live run; under a minute cached. Cost: ~$0.01. Cached reruns are free.*

## More agents is a cost, not a default

The instinct, once one agent works, is to add a second: a supervisor to coordinate, specialists for the subtasks, a manager over the specialists — an org chart drawn in prompts. Resist it as a default. Every agent you add is another model in the loop with its own context to fill, another boundary for information to fall through, and more latency and spend to absorb. Anthropic's own research feature is built as [a lead agent that coordinates the process while delegating to specialized subagents that operate in parallel](https://www.anthropic.com/engineering/multi-agent-research-system) — a real use, and one the write-up is candid about paying for in tokens. The rule that falls out: reach for a second agent when a subtask needs *isolation* or a *different toolset*, not because an org chart feels tidy.

There are two shapes worth knowing, and they differ in who runs the loop. In the first, a **supervisor** stays in charge and calls a specialist the way it calls any other tool — the specialist runs, returns a result, and control comes straight back. In the second, one agent **hands off** the whole conversation to another, which takes over the loop and runs it to the end — the framing *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation* (Wu et al., 2023, [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)) puts at the center, casting a multi-agent app as a conversation among cooperating agents. This chapter builds both against the ops desk, then measures what the second agent cost.

| Shape | Who owns the loop | Reach for it when |
|---|---|---|
| Agent-as-tool (supervisor) | the supervisor; the worker returns a value | a subtask needs isolation or a smaller, specialized toolset, and you want the result back |
| Handoff (conversational) | control transfers to the new agent | the task changes character mid-stream, and a differently-instructed agent should finish it |

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

Every model call below still routes through `shoplab.llm.complete`, so Phoenix traces both agents in a multi-agent run — the supervisor's calls and the worker's — in one project. Watching a supervisor trace fan out into a nested sub-agent is a useful second view of this chapter. Optional as ever: skip it and nothing else changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## Agent-as-tool: a specialist the supervisor can call

The first shape keeps one agent in charge. The supervisor runs the loop from chapter 02, and one of its "tools" is not a function over data but a whole other agent — its own model, its own system prompt, its own toolset — wrapped so the supervisor calls it exactly like `get_order`. The supervisor asks a question, the specialist runs to an answer, the answer comes back as a tool result. Control never leaves the supervisor.

The reason to bother is isolation. Our specialist is a *policy researcher*: it gets `search_policy` and `get_order`, nothing else. It cannot issue a refund because it holds no `issue_refund` — least privilege as a wall, not a warning. Scoping a toolset down to the few tools a role needs is chapter 09's subject; here we do it by hand — a plain dictionary filter over the standard nine, which is exactly what chapter 09 packages as `shoplab.security.scoped_tools`. Wrapping an agent as a tool is three moves: a function that runs the sub-agent, a JSON schema for its one argument, and a `Tool` around them.

In [ ]:
from dataclasses import replace
import shoplab.llm, shoplab.verify
from shoplab.llm import LEDGER
from shoplab.loop import run_agent
from shoplab.tools import Tool, standard_tools
from shoplab.world import load_tickets, load_orders, load_customers, search_policy

def scoped(names):                      # least privilege by hand; ch09 packages this as scoped_tools
    full = standard_tools()             # the standard nine
    return {n: full[n] for n in names}  # only the tools this role needs

tickets = load_tickets()
orders = {o["order_id"]: o for o in load_orders()}
customers = {c["customer_id"]: c for c in load_customers()}
ticket = next(t for t in tickets["train"] if t["ticket_id"] == "TKT-2205")

def render_ticket(t):
    return (f"Ticket {t['ticket_id']} from {t['customer_id']} about order "
            f"{t['order_id']}, sku {t['sku']}, qty {t['qty']}, condition "
            f"{t['item_condition']}, days since delivery {t['days_since_delivery']}, "
            f"photo evidence {t['evidence_photo']}, requested action "
            f"{t['requested_action']}. Customer writes: {t['reason_text']}")

In [ ]:
RESEARCHER = ("You are a policy researcher for the Larkspur Outfitters ops desk. "
              "Your only tools are search_policy and get_order; you cannot decide "
              "the ticket or move money. Answer in two or three sentences, naming "
              "the policy id you relied on and any amount.")

def make_researcher(toolset):
    def research(question):
        r = run_agent(question, toolset, system=RESEARCHER, model=MODEL, max_steps=4)
        return {"finding": r.answer, "stop": r.stop_reason}
    return research

research_tool = Tool(
    "research_policy",
    "Ask the policy researcher one question; it searches the policies and returns "
    "a finding. It cannot move money or decide the ticket.",
    {"type": "object", "properties": {"question": {"type": "string"}},
     "required": ["question"]},
    make_researcher(scoped(["search_policy", "get_order"])))

out = research_tool.fn("Order ORD-7312: opened Torrent boots returned in-window, "
                       "customer wants a refund. Which policy sets the refund, and "
                       "what amount if the customer is not vip?")
print(out["stop"], "|", out["finding"])

> **What you should see:** the researcher answers in prose — it has no `finish` tool, so the loop stops on a plain-text reply — after a few steps, and its finding names `pol-restocking` and works the restocking math: 90% of the $189.99 boot, i.e. $170.99 for a non-vip return. It reached that with a two-tool desk. It *cannot* move money, because the tool that would is not in its hand. That is the isolation an agent-as-tool buys.

## The supervisor calls it, and the trace nests

Now put the researcher to work under a supervisor whose only two tools are `research_policy` and `finish`. It cannot look anything up itself — it must delegate the lookup, read the finding, and commit to a decision. To make the nesting visible, wrap the supervisor's tools with chapter 03's `Tracer`: `research_policy` gets an `agent`-kind span, and inside it the specialist's own `search_policy` calls become child spans. One trace, two agents.

In [ ]:
from shoplab.trace import Tracer

tracer = Tracer()
def traced(toolset, kind="tool"):
    return {n: replace(t, fn=tracer.traced(n, kind=kind)(t.fn))
            for n, t in toolset.items()}

def researcher_fn(question):                     # same specialist, tools now traced
    r = run_agent(question, traced(scoped(["search_policy", "get_order"])),
                  system=RESEARCHER, model=MODEL, max_steps=4)
    return {"finding": r.answer, "stop": r.stop_reason}

fin = standard_tools()["finish"]
sup_tools = {
    "research_policy": replace(research_tool,
        fn=tracer.traced("research_policy", kind="agent")(researcher_fn)),
    "finish": replace(fin, fn=tracer.traced("finish", kind="tool")(fin.fn)),
}

In [ ]:
SUPERVISOR = ("You are the ops-desk supervisor. Your only tools are research_policy "
              "and finish -- you look nothing up yourself. Ask the specialist ONE "
              "question carrying every ticket fact, then call finish reusing the "
              "exact policy_id and amount from its finding. decision is one of "
              "approve_refund|partial_refund|replacement|store_credit|deny|escalate.")

with tracer.span("supervisor", "agent", ticket=ticket["ticket_id"]) as root:
    sup = run_agent(render_ticket(ticket), sup_tools, system=SUPERVISOR,
                    model=MODEL, max_steps=4)
    root.attributes["stop_reason"] = sup.stop_reason
print("answer:", sup.answer)
print("gold:  ", ticket["gold"])
tracer.print_tree()

> **What you should see:** the tree shows `supervisor [agent]` at the margin, a fat `research_policy [agent]` span beneath it — fat because an entire sub-agent ran inside that one tool call, its `search_policy` steps nested a level deeper — and a `finish` span last. The supervisor lands the decision *label* (`partial_refund`), but watch the other two fields: the specialist reported `pol-restocking` and `$170.99`, and the supervisor's `finish` tends to ship a different policy id or amount. Nothing corrupted the data; the finding simply degraded crossing the agent boundary, re-read and re-typed by a second model. Hold that gap — the last section puts a number on it.

## Private context, or shared: what the worker sees

When the supervisor delegates, it chooses what the worker gets to read. Two extremes. **Private**: hand the worker one focused question and nothing else — it starts from a clean, tiny context. **Shared**: forward the whole conversation so far — every lookup, every prior finding — so the worker sees what the supervisor sees. Shared feels safer, and it is not free: the worker re-reads that entire transcript on every one of its own model calls, and pays for it in tokens each time. Measure the gap on the researcher's first call, with the same question wrapped two ways.

In [ ]:
import json

question = ("Does the restocking fee apply to order ORD-7312's opened boots "
           "returned in-window, and what refund if the customer is not vip?")
history = [render_ticket(ticket), json.dumps(orders[ticket["order_id"]]),
           json.dumps(customers[ticket["customer_id"]]),
           json.dumps(search_policy("restocking opened refund returns loyalty", k=5))]
shared = "Conversation so far:\n" + "\n".join(history) + "\n\nQUESTION: " + question

def first_call_tokens(q):
    n = len(LEDGER)
    run_agent(q, scoped(["search_policy", "get_order"]),
              system=RESEARCHER, model=MODEL, max_steps=1)
    return LEDGER[n]["prompt_tokens"]

priv, shar = first_call_tokens(question), first_call_tokens(shared)
print(f"private worker prompt: {priv} tokens")
print(f"shared  worker prompt: {shar} tokens   ({shar / priv:.1f}x)")

> **What you should see:** the shared prompt runs several times larger than the private one — on our frozen run about 3x — and that multiplier lands on *every* call the worker makes, not just the first. Private context is the default for a reason: a worker handed exactly the question it must answer is cheaper and often sharper, because it is not sifting a transcript for the one fact that matters. Share context deliberately, when the worker genuinely needs the history, not reflexively.

## Handoff: transfer the conversation, don't call a subordinate

The second shape is not a subordinate you call and get a value back from — it is a *transfer*. One agent runs until it decides the task belongs to someone else, then hands the whole conversation forward to a differently-instructed agent that takes over the loop and finishes it. This is the AutoGen framing: cooperating agents passing a conversation between them, not a caller and a callee.

Our triager is a read-only agent — lookups and `finish`, no write tools. Its instructions: judge the ticket, but on any fraud or abuse signal, do *not* decide it — end with the word `HANDOFF`. When it does, the router forwards its message list to an escalation agent that has a different system prompt and the `escalate` tool the triager never held. The difference from agent-as-tool is who owns the loop: here the triager is *done*, and the escalation agent runs to the end on its own.

In [ ]:
esc_ticket = next(t for t in tickets["train"] if t["ticket_id"] == "TKT-2209")

TRIAGE = ("You are the ops-desk triager. Look up the order and the customer, then "
          "judge the ticket. On a fraud or abuse signal -- a flagged serial "
          "returner, a damaged/defective claim over $75 with no photo, or a large "
          "unevidenced claim -- do NOT decide it: end your reply with the word "
          "HANDOFF. Otherwise call finish with the decision.")

tri = run_agent(render_ticket(esc_ticket),
                scoped(["get_order", "get_customer", "search_policy", "finish"]),
                system=TRIAGE, model=MODEL, max_steps=5)
hands_off = isinstance(tri.answer, str) and "HANDOFF" in tri.answer.upper()
print("triager stop:", tri.stop_reason, "| hands off:", hands_off)

In [ ]:
def forward(messages):
    lines = []
    for m in messages:
        if m.get("role") == "system":
            continue
        if m.get("content"):
            lines.append(f"{m['role']}: {m['content']}")
        for tc in m.get("tool_calls") or []:
            f = tc["function"]
            lines.append(f"assistant -> {f['name']}({f['arguments']})")
    return "\n".join(lines)

In [ ]:
ESCALATION = ("You are the fraud-and-escalation desk. A triager handed you the "
              "conversation below. Review it, call escalate with the ticket id and "
              "a one-line risk note, then call finish with decision 'escalate', "
              "policy_id 'pol-fraud', refund_usd null.")

handed = forward(tri.messages)
esc = run_agent("Ticket " + esc_ticket["ticket_id"] + ", conversation so far:\n" + handed,
                scoped(["get_order", "get_customer", "escalate", "finish"]),
                system=ESCALATION, model=MODEL, max_steps=5)
print("forwarded", len(handed), "chars ->", handed[:70], "...")
print("escalation:", esc.answer)
print("gold:      ", esc_ticket["gold"])

> **What you should see:** on this high-value, no-photo damage claim the triager reads the order and customer, recognizes an unevidenced claim over $300, and ends with `HANDOFF` rather than deciding — `hands off` is `True`. The router forwards the transcript, and the escalation agent — now holding `escalate` — reviews it and finishes with `escalate` / `pol-fraud`, matching gold. Two agents, one conversation: the triager owned the loop until the handoff, then the escalation agent owned it. Contrast the agent-as-tool run, where control snapped back to the supervisor after every specialist call.

## When the second agent is a tax

Both shapes work. The question the thesis demands is whether they *earn their keep* on a task a single agent already handles — and TKT-2205 is exactly that task. Run the whole nine-tool desk on it as one agent, then run the supervisor+worker split on the same ticket, and count both: model calls, tokens, dollars, and — the field that actually matters — whether the answer agrees with the rules engine, using chapter 06's free deterministic `check_decision`.

In [ ]:
SYSTEM_FULL = ("You are the operations desk agent for Larkspur Outfitters. Look up "
               "the order, the customer, and the relevant policy, compute any "
               "amount with calc, then call finish with decision, policy_id, and "
               "refund_usd.")
order, customer = orders[ticket["order_id"]], customers[ticket["customer_id"]]

def cost_of(run):
    n = len(LEDGER)
    ans = run()
    rows = LEDGER[n:]
    return {"calls": len(rows),
            "tokens": sum(r["prompt_tokens"] + r["completion_tokens"] for r in rows),
            "cost": round(sum(r["cost_usd"] for r in rows), 5),
            "agrees": shoplab.verify.check_decision(ans, ticket, order, customer)["agrees"],
            "answer": ans}

In [ ]:
single = cost_of(lambda: run_agent(render_ticket(ticket), standard_tools(),
                                   system=SYSTEM_FULL, model=MODEL, max_steps=8).answer)
multi = cost_of(lambda: run_agent(render_ticket(ticket),
               {"research_policy": research_tool, "finish": standard_tools()["finish"]},
               system=SUPERVISOR, model=MODEL, max_steps=4).answer)
for name, r in [("single agent", single), ("supervisor+worker", multi)]:
    print(f"{name:18} calls={r['calls']:2} tokens={r['tokens']:5} "
          f"cost=${r['cost']:.5f} agrees={r['agrees']}  {r['answer']}")

> **What you should see:** the single agent, with the whole desk in one context, agrees with the rules engine — right decision, policy id, and amount. The supervisor+worker split does not: it fumbles the policy id and the amount at the boundary the trace exposed earlier, so `agrees` is `False`. Read the numbers carefully. The split may even spend *fewer* tokens getting there — a two-tool supervisor and a two-tool worker each carry a smaller schema than the nine-tool desk — but that is no win: it bought a *wrong* refund for the saving, and the saving is a mirage. The supervisor delegated just once here; each extra `research_policy` call drags in a whole sub-agent (the fat span from earlier), so a second delegation erases the gap. Cheaper-but-wrong is the most expensive outcome on a desk that moves money. That is the tax: pay it only when isolation or a different toolset is worth more than the fidelity you lose at the seam.

## When a second agent earns its keep

The failure above was a task that never needed splitting. The wins were the escalation handoff — a genuinely different job, with different tools and instructions — and the scoped researcher, a subtask walled off from the money tools. That is the pattern: a second agent pays when the subtask is *different in kind*, not merely another step of the same work.

| Reach for | When |
|---|---|
| One agent | the whole task fits one toolset and one context; this is the default |
| Agent-as-tool | a subtask needs isolation (least privilege) or a smaller, specialized toolset, and you want a result back |
| Handoff | the task changes character mid-stream — a different role, prompt, and tools should run it to completion |
| More agents still | only when subtasks genuinely parallelize or must not share context, and you have measured that one agent cannot cope |

## Recap

| Concept | One-liner |
|---|---|
| More agents is a cost | each added agent is another context to fill, more latency, and another boundary a finding degrades crossing — spend that is easy to multiply and hard to predict. |
| Agent-as-tool | wrap a whole sub-agent as one `Tool`; the supervisor calls it and control returns with a value. |
| Least-privilege specialist | give the worker only the tools its role needs — it cannot misuse a tool it does not hold (chapter 09 packages this as `scoped_tools`). |
| Nested trace | an `agent`-kind span whose children are the sub-agent's own tool calls — one trace, two agents. |
| Private vs shared context | a focused question is cheap; forwarding the whole transcript costs its tokens on every worker call. |
| Handoff | transfer the conversation to a differently-instructed agent that takes over the loop and finishes it. |
| Who owns the loop | agent-as-tool returns control to the caller; a handoff gives it away. |
| The lossy boundary | a finding re-read by a second model degrades; measure `agrees`, not vibes, before you split. |

## Exercises

1. Add a second specialist — an `arithmetic` agent scoped to `calc` alone — and give the supervisor both it and `research_policy`. Does letting the supervisor delegate the math, instead of transcribing the amount out of the researcher's prose, fix the `refund_usd` drift from the last section, and what does it cost in extra calls?
2. Measure context, not guesses. For the same ticket, run the researcher once with a private question and once with the full conversation forwarded, and sum the prompt tokens over *all* the worker's calls, not just the first. Tabulate private vs shared against how many steps the worker takes — where does shared context stop being affordable?
3. Convert the handoff into an agent-as-tool. Wrap the escalation agent as a `Tool` the triager can call, so the triager keeps the loop and gets a result back instead of transferring control. Run both on TKT-2209: does the outcome change, and which shape leaves a cleaner trace to audit when a refund is later disputed?

**Next up:** chapter 08 puts guardrails on all of this — real budgets that cap calls and dollars, checkpoints that resume a crashed run, and approval gates that stop a risky tool before it fires.